In [16]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import Dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

text = Path("tiny-shakespeare.txt").read_text()

In [5]:
#print(text[:1000])

In [24]:
class CharactTokenizer:
    def __init__(self, vocabulary):
        self.token_id_for_char = {
            char: token_id for token_id, char in enumerate(vocabulary)
        }
        self.char_for_token_id = {
            token_id: char for token_id, char in enumerate(vocabulary)
        }

    @staticmethod
    def train_from_text(text):
        vocabulary = set(text)
        return CharactTokenizer(sorted(list(vocabulary)))

    def encode(self, text):
        token_ids = []
        for char in text:
            token_ids.append(self.token_id_for_char[char])
        return torch.tensor(token_ids, dtype=torch.long)

    def decode(self, token_ids):
        chars = []
        for token_id in token_ids.tolist():
            chars.append(self.char_for_token_id[token_id])
        return ''.join(chars)

    def vocabulary_size(self):
        return len(self.token_id_for_char)


class TokenIdsDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, pos):
        assert pos < len(self.data) - self.block_size
        x = self.data[pos: pos + self.block_size]
        y = self.data[pos + 1 : pos + 1 + self.block_size]
        return x, y

In [23]:
tokenizer = CharactTokenizer.train_from_text(text)
tokenizer
print(tokenizer.encode("Hello Green"))
print(tokenizer.decode(tokenizer.encode("Hello Green")))
print(tokenizer.vocabulary_size())

tensor([20, 43, 50, 50, 53,  1, 19, 56, 43, 43, 52])
Hello Green
65


In [25]:
config = {
    "vocabulary_size": tokenizer.vocabulary_size(),
    "context_size": 256,
    "d_embed": 768,
    "heads_num": 12,
    "layers_num": 10,
    "dropout_rate": 0.1,
    "use_bias": False,
}

config["head_size"] = config["d_embed"] // config["heads_num"]

config

{'vocabulary_size': 65,
 'context_size': 256,
 'd_embed': 768,
 'heads_num': 12,
 'layers_num': 10,
 'dropout_rate': 0.1,
 'use_bias': False,
 'head_size': 64}

In [33]:
class AttentionHead(nn.Module):
    def __init__(self, config):
        super().__init__()
        
        self.Q_weights = nn.Linear(
            config["d_embed"], config["head_size"], config["use_bias"]
        )
        self.K_weights = nn.Linear(
            config["d_embed"], config["head_size"], config["use_bias"]
        )
        self.V_weights = nn.Linear(
            config["d_embed"], config["head_size"], config["use_bias"]
        )

        self.dropout = nn.Dropout(
            config["dropout_rate"]
        )

        casual_attention_mask = torch.tril(
            torch.ones(config["context_size"], config["context_size"])
        )

        self.register_buffer("casual_attention_mask", casual_attention_mask)

    def forward(self, input):
        batch_size, tokens_num, d_embed = input.shape
        Q = self.Q_weights(input)
        K = self.K_weights(input)
        V = self.V_weights(input)

        attention_scores = Q @ K.transpose(1,2)
        attention_scores = attention_scores.masked_fill(
            self.casual_attention_mask[:tokens_num, :tokens_num] == 0, -torch.inf
        )

        attention_scores = attention_scores / (K.shape[-1] ** 0.5)
        attention_scores = torch.softmax(attention_scores, dim=1)
        attention_scores = self.dropout(attention_scores)

        return attention_scores @ V

In [43]:
input = torch.rand(8, config['context_size'], config['d_embed'])

In [44]:
ah = AttentionHead(config)

In [45]:
ah

AttentionHead(
  (Q_weights): Linear(in_features=768, out_features=64, bias=False)
  (K_weights): Linear(in_features=768, out_features=64, bias=False)
  (V_weights): Linear(in_features=768, out_features=64, bias=False)
  (dropout): Dropout(p=0.1, inplace=False)
)

In [46]:
output = ah(input)
print(output)

tensor([[[-2.7916e-03, -1.5587e-03, -2.3693e-04,  ...,  1.9088e-04,
           1.4772e-03,  1.1492e-03],
         [-4.4198e-03, -3.0011e-03, -2.5401e-03,  ...,  6.4694e-04,
           2.5964e-03,  1.1121e-03],
         [-6.4409e-03, -5.1450e-03, -3.3600e-03,  ..., -1.5331e-04,
           4.4114e-03,  1.6432e-03],
         ...,
         [-2.0265e+00, -1.2815e+00, -1.8064e+00,  ..., -2.1459e-02,
           1.6530e+00,  5.9562e-01],
         [-2.4179e+00, -1.4825e+00, -1.9918e+00,  ..., -2.2114e-01,
           1.7212e+00,  7.6443e-01],
         [-3.0163e+00, -1.8809e+00, -2.6530e+00,  ..., -1.5449e-01,
           1.9981e+00,  6.3434e-01]],

        [[-2.0720e-03, -2.8131e-03, -1.5460e-03,  ..., -6.3685e-04,
           1.8446e-03,  1.5731e-04],
         [-3.6088e-03, -4.0736e-03, -3.3575e-03,  ..., -3.3811e-04,
           2.9329e-03,  9.4802e-04],
         [-4.2892e-03, -3.9476e-03, -5.4371e-03,  ..., -5.7756e-04,
           4.1542e-03,  2.2601e-03],
         ...,
         [-2.1568e+00, -1

In [47]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()

        heads_list = [AttentionHead(config) for _ in range(config["heads_num"])]
        self.heads = nn.ModuleList(heads_list)

        self.linear = nn.Linear(config["d_embed"], config["d_embed"])
        self.dropout = nn.Dropout(config["dropout_rate"])

    def forward(self, input):
        heads_outputs = [head(input) for head in self.heads]

        scores_change = torch.cat(heads_outputs, dim=-1)
        scores_change = self.linear(scores_change)

        return self.dropout(scores_change)

In [48]:
mha = MultiHeadAttention(config)

In [49]:
input = torch.rand(8, config["context_size"], config["d_embed"])

In [50]:
output = mha(input)

In [52]:
output.shape

torch.Size([8, 256, 768])

In [56]:
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.linear_layers = nn.Sequential(
            nn.Linear(config["d_embed"], config["d_embed"] * 4),
            nn.GELU(),
            nn.Linear(config["d_embed"] * 4, config["d_embed"]),
            nn.Dropout(config["dropout_rate"]),
        )

    def forward(self, input):
        return self.linear_layers(input)

In [57]:
ff = FeedForward(config)

In [58]:
input = torch.rand(8, config["context_size"], config["d_embed"])

In [61]:
output = ff(input)
output

tensor([[[-0.0609,  0.1845, -0.0452,  ...,  0.0000, -0.0232, -0.1337],
         [-0.0447,  0.1227, -0.0088,  ...,  0.1057,  0.0078, -0.1653],
         [-0.0558,  0.2093, -0.0143,  ...,  0.0727,  0.1467, -0.1529],
         ...,
         [-0.1226,  0.0243, -0.0291,  ...,  0.0766,  0.0936, -0.0000],
         [-0.1758,  0.2018, -0.0853,  ..., -0.0577,  0.0446, -0.1695],
         [-0.1328,  0.0586, -0.0501,  ...,  0.1363, -0.0786, -0.1891]],

        [[-0.1216,  0.1500, -0.0484,  ...,  0.0000, -0.0273, -0.2344],
         [-0.0769,  0.1453,  0.0063,  ...,  0.1445, -0.0425, -0.1987],
         [-0.0000,  0.0512, -0.0315,  ...,  0.0490,  0.0079, -0.2629],
         ...,
         [-0.0000,  0.1049,  0.0215,  ...,  0.0833, -0.0298, -0.0986],
         [-0.0000,  0.0000, -0.0656,  ...,  0.0953,  0.0733, -0.2493],
         [-0.1242,  0.1143,  0.0270,  ...,  0.1024, -0.0355, -0.1573]],

        [[-0.1193,  0.0000, -0.0372,  ...,  0.1060, -0.0107, -0.1143],
         [-0.0550,  0.1628,  0.0834,  ...,  0

In [69]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.multi_head = MultiHeadAttention(config)
        self.layer_norm_1 = nn.LayerNorm(config["d_embed"])

        self.feed_forward = FeedForward(config)
        self.layer_norm_2 = nn.LayerNorm(config["d_embed"])

    def forward(self, input):
        residual = input
        x = self.multi_head(self.layer_norm_1(input))
        x = x + residual

        residual = x
        x = self.feed_forward(self.layer_norm_2(x))
        return x + residual

In [70]:
b = Block(config)

In [71]:
b

Block(
  (multi_head): MultiHeadAttention(
    (heads): ModuleList(
      (0-11): 12 x AttentionHead(
        (Q_weights): Linear(in_features=768, out_features=64, bias=False)
        (K_weights): Linear(in_features=768, out_features=64, bias=False)
        (V_weights): Linear(in_features=768, out_features=64, bias=False)
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (linear): Linear(in_features=768, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (layer_norm_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (feed_forward): FeedForward(
    (linear_layers): Sequential(
      (0): Linear(in_features=768, out_features=3072, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=3072, out_features=768, bias=True)
      (3): Dropout(p=0.1, inplace=False)
    )
  )
  (layer_norm_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)

In [72]:
output.shape

torch.Size([8, 256, 768])

In [73]:
class DemoGPT(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.token_embedding_layer = nn.Embedding(
            config["vocabulary_size"], config["d_embed"]
        )
        self.positional_embedding_layer = nn.Embedding(
            config["context_size"], config["d_embed"]
        )

        blocks = [Block(config) for _ in range(config["layers_num"])]
        self.layers = nn.Sequential(*blocks)

        self.layer_norm = nn.LayerNorm(config["d_embed"])
        self.unembedding = nn.Linear(
            config["d_embed"], config["vocabulary_size"], bias=False
        )

    def forward(self, token_ids):
        batch_size, tokens_num = token_ids.shape

        x = self.token_embedding_layer(token_ids)
        sequence = torch.arange(tokens_num, device=device)
        x = x + self.positional_embedding_layer(sequence)

        x = self.layers(x)
        x = self.layer_norm(x)
        x = self.unembedding(x)

        return x

In [74]:
model = DemoGPT(config).to(device)

In [75]:
output = model(tokenizer.encode("Hi").unsqueeze(dim=0).to(device))

In [76]:
output.shape

torch.Size([1, 2, 65])

In [90]:
def generate(model, prompt_ids, max_tokens):
    output_ids = prompt_ids
    for _ in range(max_tokens):
        if output_ids.shape[1] >= config["context_size"]:
            break
        with torch.no_grad():
            logps = model(output_ids)
        logps = logps[:, -1, :]
        probs = F.softmax(logps, dim=-1)
        next_token_id = torch.multinomial(probs, num_samples=1)
        output_ids = torch.cat([output_ids, next_token_id], dim=-1)
    return output_ids

In [103]:
def generate_with_prompt(model, tokenizer, prompt, max_tokens=100):
    model.eval()
    prompt = tokenizer.encode(prompt).unsqueeze(dim=0).to(device)

    return tokenizer.decode(generate(model, prompt, max_tokens=max_tokens)[0])

In [106]:
generate_with_promt(model, tokenizer, "First Class:\n")

"First Class:\n n     '      e      e                  F          e        de                    r          o      "

In [107]:
batch_size = 64

train_iterations = 500
evaluation_interval = 10
learning_rate = 4e-4
train_split = 0.9

In [108]:
tokenized_text = tokenizer.encode(text).to(device)
train_count = int(train_split * len(tokenized_text))
train_data, validation_data = tokenized_text[:train_count], tokenized_text[train_count:]

In [109]:
train_dataset = TokenIdsDataset(train_data, config["context_size"])
validation_dataset = TokenIdsDataset(validation_data, config["context_size"])

In [110]:
from torch.utils.data import Dataset, DataLoader, RandomSampler

train_sampler = RandomSampler(
    train_dataset, num_samples=batch_size * train_iterations, replacement=True
)
train_dataloader = DataLoader(
    train_dataset, batch_size=batch_size, sampler=train_sampler
)

validation_sampler = RandomSampler(validation_dataset, replacement=True)
validation_dataloader = DataLoader(
    validation_dataset, batch_size=batch_size, sampler=validation_sampler
)

In [111]:
@torch.no_grad()
def calculate_validation_loss(model, batches_num):
    model.eval()
    total_loss = 0

    validation_iter = iter(validation_dataloader)

    for _ in range(batches_num):
        input, targets = next(validation_iter)
        logits = model(input)

        logits_view = logits.view(
            batch_size * config["context_size"], config["vocabulary_size"]
        )
        targets_view = targets.view(batch_size * config["context_size"])

        loss = F.cross_entropy(logits_view, targets_view)

        total_loss += loss.item()

    average_loss = total_loss / batches_num

    return average_loss

In [112]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [113]:
import os
from IPython.display import display, clear_output
from matplotlib import pyplot as plt
from IPython.display import display
import ipywidgets as widgets
%matplotlib inline

plot_output = widgets.Output()

display(plot_output)

def update_plot(train_losses, train_steps, validation_losses, validation_steps):

  with plot_output:
    clear_output(wait=True)  # Clear only the plot output, not the text
    plt.figure(figsize=(7, 5))
    plt.plot(train_steps, train_losses, label='Training Loss')
    plt.plot(validation_steps, validation_losses, label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('epoch')
    plt.legend(loc='center left')
    plt.grid(True)
    plt.show()


# Set up lists to store losses for plotting
train_losses = []
train_steps = []
eval_losses = []
eval_steps = []


for step_num, sample in enumerate(train_dataloader):

  model.train()
  input, targets = sample
  logits = model(input)

  logits_view = logits.view(batch_size * config["context_size"], config["vocabulary_size"])
  targets_view = targets.view(batch_size * config["context_size"])
  
  loss = F.cross_entropy(logits_view, targets_view)
  # Backward propagation
  loss.backward()
  # Update model parameters
  optimizer.step()
  # Set to None to reduce memory usage
  optimizer.zero_grad(set_to_none=True)

  train_losses.append(loss.item())
  train_steps.append(step_num)

  print(f"Step {step_num}. Loss {loss.item():.3f}")

  if step_num % evaluation_interval == 0:
    print("Demo GPT:\n" + generate_with_prompt(model, tokenizer, "\n"))

    validation_loss = calculate_validation_loss(model, batches_num=10)
    eval_losses.append(validation_loss)
    eval_steps.append(step_num)

    print(f"Step {step_num}. Validation loss: {validation_loss:.3f}")


  update_plot(train_losses, train_steps, eval_losses, eval_steps)

Output()

Step 0. Loss 4.270
Demo GPT:

eree
Toeoreondeerogd',e?peeeesetesgsese.oygehs!aiooheeeeerogmuofoo'bMsfereoomOedCVoenoarewlk
enemeem
Step 0. Validation loss: 4.591
Step 1. Loss 4.579
Step 2. Loss 4.544
Step 3. Loss 3.957
Step 4. Loss 3.671
Step 5. Loss 3.196
Step 6. Loss 3.035
Step 7. Loss 2.973
Step 8. Loss 2.840
Step 9. Loss 2.866
Step 10. Loss 2.782
Demo GPT:

H ito ocoald sHa
uIipoOoA:
Solh:

on tuu rare&wi krit ho,orpdeveo,
AILToul.
p.
UrisFaglae,O:
H;owauh
Step 10. Validation loss: 2.748
Step 11. Loss 2.741
Step 12. Loss 2.693
Step 13. Loss 2.728
Step 14. Loss 2.686
Step 15. Loss 2.636
Step 16. Loss 2.622
Step 17. Loss 2.610
Step 18. Loss 2.600
Step 19. Loss 2.612
Step 20. Loss 2.589
Demo GPT:

NTE:
ENG-ilin k  her
ILGitonskke
TThensbd h s aa fe heththen a;

Mie fk, w he te bkeaseak stonum ho 
Step 20. Validation loss: 2.591
Step 21. Loss 2.590
Step 22. Loss 2.574
Step 23. Loss 2.557
Step 24. Loss 2.548
Step 25. Loss 2.525
Step 26. Loss 2.553
Step 27. Loss 2.526
Step 28. Loss 2.539